In [2]:
import os  # noqa: F401

from langchain_classic.retrievers import ContextualCompressionRetriever  # noqa: F401
from langchain_classic.retrievers.document_compressors import LLMChainExtractor  # noqa: F401
from langchain_community.vectorstores import FAISS  # noqa: F401
from langchain_core.documents import Document  # noqa: F401
from langchain_openai import ChatOpenAI, OpenAIEmbeddings  # noqa: F401


In [5]:
documents = [
    Document(
        page_content=(
            "The Amazon rainforest supports an extraordinary variety of plants, animals, "
            "and insects. Its dense forests provide shelter and food for countless species, "
            "many of which cannot survive outside this ecosystem.\n\n"
            "The rainforest also supports biodiversity through its rivers, wetlands, and "
            "layered vegetation. Protecting these habitats helps preserve endangered species "
            "and maintains the ecological balance of the region."
        ),
        metadata={"topic": "environment", "source": "doc_1"},
    ),
    Document(
        page_content=(
            "Italian cuisine is strongly influenced by regional ingredients and cooking "
            "traditions. Northern regions commonly use rice, butter, and cheese, while "
            "southern regions often rely on tomatoes, olive oil, seafood, and herbs.\n\n"
            "These regional ingredients shape the preparation of pasta, soups, breads, and "
            "main dishes. Italian cooks usually emphasize fresh seasonal products and simple "
            "methods that preserve the natural flavor of the food."
        ),
        metadata={"topic": "food", "source": "doc_2"},
    ),
    Document(
        page_content=(
            "The Great Wall of China was constructed by different dynasties to defend "
            "settlements and important borders. Its walls, towers, and gates helped control "
            "movement through mountain passes and trade routes.\n\n"
            "Military guards used the wall as a defensive network rather than as one continuous "
            "barrier. Watchtowers allowed soldiers to observe distant activity and send warnings "
            "using smoke, fire, flags, and other visual signals."
        ),
        metadata={"topic": "history", "source": "doc_3"},
    ),
    Document(
        page_content=(
            "Honeybees live in colonies where each bee has a specific role. The queen lays "
            "eggs, worker bees gather nectar and pollen, and drones help reproduce with the queen. "
            "This division of labor keeps the colony functioning.\n\n"
            "Worker bees also protect and maintain the hive while caring for developing young. "
            "Their collection of nectar and pollen supports the colony's food supply and allows "
            "bees to continue raising new generations."
        ),
        metadata={"topic": "nature", "source": "doc_4"},
    ),
    Document(
        page_content=(
            "Regular physical activity strengthens the heart, muscles, and bones while improving "
            "balance and endurance. Activities such as walking, swimming, cycling, and dancing "
            "can provide health benefits when practiced consistently.\n\n"
            "Exercise is more effective when it becomes part of a sustainable routine. Starting "
            "with manageable activities and gradually increasing their duration can help people "
            "remain active without placing excessive strain on the body."
        ),
        metadata={"topic": "wellness", "source": "doc_5"},
    ),
]

print(len(documents))


5


In [4]:
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large",
    dimensions=30,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

vector_store = FAISS.from_documents(documents, embedding_model)

In [6]:
base_retriever = vector_store.as_retriever(search_kwargs={"k": 5})

In [9]:
# set up the LLM for the multi-query retriever

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
)
compressor = LLMChainExtractor.from_llm(
    llm=llm
)

In [11]:
# set up the contextual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever,
)

In [13]:
# query the contextual compression retriever
query = "What are the health benefits of regular physical activity?"
results = compression_retriever.invoke(query)

In [14]:
for i, result in enumerate(results):
    print(f"Result {i+1}:")
    print(f"Content: {result.page_content}")
    print(f"Source: {result.metadata['source']}")
    print("-" * 50)


Result 1:
Content: Regular physical activity strengthens the heart, muscles, and bones while improving balance and endurance. Activities such as walking, swimming, cycling, and dancing can provide health benefits when practiced consistently.
Source: doc_5
--------------------------------------------------


In [15]:
query2 = "Give me a summary of the key points about the Amazon rainforest's biodiversity and its importance to the ecosystem."
results2 = compression_retriever.invoke(query2)

In [16]:
print("Results for Query 2:")
for i, result in enumerate(results2):
    print(f"Result {i+1}:")
    print(f"Content: {result.page_content}")
    print(f"Source: {result.metadata['source']}")
    print("-" * 50)

Results for Query 2:
Result 1:
Content: The Amazon rainforest supports an extraordinary variety of plants, animals, and insects. Its dense forests provide shelter and food for countless species, many of which cannot survive outside this ecosystem.

The rainforest also supports biodiversity through its rivers, wetlands, and layered vegetation. Protecting these habitats helps preserve endangered species and maintains the ecological balance of the region.
Source: doc_1
--------------------------------------------------
